In [1]:
import pandas as pd
import warnings
from matplotlib import MatplotlibDeprecationWarning
warnings.filterwarnings("ignore", category=MatplotlibDeprecationWarning)

import sys
from transformers import DistilBertTokenizer
sys.path.append("../../../splade_sans/learned-sparse-retrieval-1.0.0")
from lsr.transformer import LSR
from IPython.display import display, HTML
from matplotlib.colors import Normalize, to_hex
from matplotlib import cm
import numpy as np
import pyterrier as pt
if not pt.started():
    pt.init()

/tmp/ipykernel_502/359739181.py:15: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_502/359739181.py:16: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [2]:
ohsumed_docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

In [3]:
def get_colored_tokens_html(tokens, scores, colour="Greens"):
    """
    Returns an HTML string with tokens color-coded by importance.
    """
    # Normalize scores between 0 and 1
    norm = Normalize(vmin=min(scores), vmax=max(scores))
    if colour == "green":
        cmap = cm.get_cmap("Greens")
    elif colour == "red":
        cmap = cm.get_cmap("Reds")
        
    colored_tokens = []
    for token, score in zip(tokens, scores):
        color = to_hex(cmap(norm(score)))
        span = f'<span style="background-color:{color}; padding:2px; border-radius:4px;">{token}</span>'
        colored_tokens.append(span)

    return " ".join(colored_tokens)

# def get_colored_tokens_latex(tokens, scores):

#     from matplotlib import cm
#     from matplotlib.colors import Normalize, rgb2hex

#     norm = Normalize(vmin=min(scores), vmax=max(scores))
#     cmap = cm.get_cmap("Reds")

#     colored_tokens = []
#     for token, score in zip(tokens, scores):
#         # Convert normalized score to RGB, then to xcolor-compatible format
#         rgba = cmap(norm(score))[:3]
#         r, g, b = [int(c * 255) for c in rgba]
#         color_str = f"[RGB]{{{r},{g},{b}}}"

#         box = f"\\colorbox{color_str}{{{token}}}"
#         colored_tokens.append(box)

#     return " ".join(colored_tokens)

def get_colored_tokens_latex(tokens, scores, colormap="tableau-colorblind10"):

    from matplotlib import cm
    from matplotlib.colors import Normalize

    norm = Normalize(vmin=min(scores), vmax=max(scores))
    cmap = cm.get_cmap(colormap)

    colored_tokens = []
    for token, score in zip(tokens, scores):
        rgba = cmap(norm(score))[:3]  # Get R, G, B tuple
        r, g, b = [int(255 * c) for c in rgba]
        color = f"[RGB]{{{r},{g},{b}}}"

        box = f"\\colorbox{color}{{{token}}}"
        colored_tokens.append(box)

    return " ".join(colored_tokens)

In [4]:
distil_model_path = "/nfs/primary/SPLADE/splade_class/lsr_package/outputs/hn_t5_retrain_scratch/model"
distil_model = LSR(distil_model_path)

relevance_model_path = "/nfs/primary/SPLADE/splade_class/lsr_package/outputs/splade_ohsumed_multiple_negative/model"
relevance_model = LSR(relevance_model_path)

In [21]:
def get_tokens_and_intensities(query, document, model):
    q_rep = model.encode_queries([query], out_fmt = "np")[0]
    d_rep = model.encode_docs([document], out_fmt = "np")[0]

    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    # Tokenize using the model's tokenizer
    q_tokens = tokenizer.tokenize(query)
    d_tokens = tokenizer.tokenize(document)

    q_token_ids = tokenizer.convert_tokens_to_ids(q_tokens)
    d_token_ids = tokenizer.convert_tokens_to_ids(d_tokens)

    # Collect intensity values from representation vector
    q_token_values = [q_rep[token_id] for token_id in q_token_ids]
    d_token_values = [d_rep[token_id] for token_id in d_token_ids]

    
    return (q_tokens, q_token_values), (d_tokens, d_token_values)

In [6]:
def display_colours(text, model, colour = "green"):
    out = get_tokens_and_intensities(text, "paris is the main airport", model)[0]
    html = get_colored_tokens_html(out[0], out[1], colour)
    display(HTML(html))
    # print(get_colored_tokens_latex(out[0], out[1]))
    

# for i in range(50):
#     display_colours(ohsumed_docs[ohsumed_docs["sensitivity"] == 0].head(100).text.iloc[i], relevance_model)
#     print()
#     display_colours(ohsumed_docs[ohsumed_docs["sensitivity"] == 0].head(100).text.iloc[i], distil_model)
    

#     print("-" * 1000)
#     print()

In [7]:
sensitive_doc = ohsumed_docs[ohsumed_docs.apply(lambda row: row.astype(str).str.contains('How often does maternal').any(), axis=1)].text.tolist()[0]
non_sensitive_doc = ohsumed_docs[ohsumed_docs.apply(lambda row: row.astype(str).str.contains('disorder reported in the past was associated with little or no bleeding').any(), axis=1)].text.tolist()[0]

In [20]:
display_colours(sensitive_doc, relevance_model, "red")
display_colours(sensitive_doc, distil_model, "red")
print()
print()
display_colours(non_sensitive_doc, relevance_model, "green")
display_colours(non_sensitive_doc, distil_model, "green")

['how', 'often', 'does', 'maternal', 'pre', '##ec', '##lam', '##ps', '##ia', '-', 'ec', '##lam', '##ps', '##ia', 'inc', '##ite', 'th', '##rom', '##bo', '##cy', '##top', '##enia', 'in', 'the', 'fe', '##tus', '?', '[', 'letter', ']', 'to', 'sim', '##plify', 'the', 'procedure', 'for', 'obtaining', 'individual', 'growth', 'curve', 'standards', ',', 'we', 'investigated', 'the', 'use', 'of', 'models', 'derived', 'from', 'the', 'slopes', 'of', 'growth', 'curves', 'determined', 'before', '26', 'weeks', "'", 'men', '##st', '##ru', '##al', 'age', '.', 'in', 'serial', 'examinations', 'of', '33', 'normal', 'fe', '##tus', '##es', 'between', '15', '-', '38', 'weeks', ',', 'we', 'obtained', 'measurements', 'of', 'the', 'bi', '##par', '##ie', '##tal', 'diameter', '(', 'bp', '##d', ')', ',', 'head', 'ci', '##rc', '##um', '##ference', ',', 'abdominal', 'ci', '##rc', '##um', '##ference', ',', 'fe', '##mur', 'dia', '##phy', '##sis', 'length', ',', 'head', 'profile', 'area', ',', 'abdominal', 'profile', 'a

['how', 'often', 'does', 'maternal', 'pre', '##ec', '##lam', '##ps', '##ia', '-', 'ec', '##lam', '##ps', '##ia', 'inc', '##ite', 'th', '##rom', '##bo', '##cy', '##top', '##enia', 'in', 'the', 'fe', '##tus', '?', '[', 'letter', ']', 'to', 'sim', '##plify', 'the', 'procedure', 'for', 'obtaining', 'individual', 'growth', 'curve', 'standards', ',', 'we', 'investigated', 'the', 'use', 'of', 'models', 'derived', 'from', 'the', 'slopes', 'of', 'growth', 'curves', 'determined', 'before', '26', 'weeks', "'", 'men', '##st', '##ru', '##al', 'age', '.', 'in', 'serial', 'examinations', 'of', '33', 'normal', 'fe', '##tus', '##es', 'between', '15', '-', '38', 'weeks', ',', 'we', 'obtained', 'measurements', 'of', 'the', 'bi', '##par', '##ie', '##tal', 'diameter', '(', 'bp', '##d', ')', ',', 'head', 'ci', '##rc', '##um', '##ference', ',', 'abdominal', 'ci', '##rc', '##um', '##ference', ',', 'fe', '##mur', 'dia', '##phy', '##sis', 'length', ',', 'head', 'profile', 'area', ',', 'abdominal', 'profile', 'a



['pulmonary', 'em', '##bol', '##us', '-', 'induced', 'di', '##sse', '##minated', 'intra', '##vas', '##cular', 'coa', '##gul', '##ation', '.', 'pulmonary', 'em', '##bol', '##us', 'as', 'a', 'cause', 'of', 'di', '##sse', '##minated', 'intra', '##vas', '##cular', 'coa', '##gul', '##ation', 'has', 'only', 'recently', 'been', 'recognized', '.', 'the', 'hem', '##or', '##rh', '##agi', '##c', 'disorder', 'reported', 'in', 'the', 'past', 'was', 'associated', 'with', 'little', 'or', 'no', 'bleeding', '.', 'we', 'report', 'a', 'case', 'of', 'pulmonary', 'em', '##bol', '##us', 'associated', 'with', 'life', '-', 'threatening', 'di', '##sse', '##minated', 'intra', '##vas', '##cular', 'coa', '##gul', '##ation', '.']
[21908, 7861, 14956, 2271, 1011, 10572, 4487, 11393, 26972, 26721, 12044, 15431, 28155, 24848, 3370, 1012, 21908, 7861, 14956, 2271, 2004, 1037, 3426, 1997, 4487, 11393, 26972, 26721, 12044, 15431, 28155, 24848, 3370, 2038, 2069, 3728, 2042, 3858, 1012, 1996, 19610, 2953, 25032, 22974, 

['pulmonary', 'em', '##bol', '##us', '-', 'induced', 'di', '##sse', '##minated', 'intra', '##vas', '##cular', 'coa', '##gul', '##ation', '.', 'pulmonary', 'em', '##bol', '##us', 'as', 'a', 'cause', 'of', 'di', '##sse', '##minated', 'intra', '##vas', '##cular', 'coa', '##gul', '##ation', 'has', 'only', 'recently', 'been', 'recognized', '.', 'the', 'hem', '##or', '##rh', '##agi', '##c', 'disorder', 'reported', 'in', 'the', 'past', 'was', 'associated', 'with', 'little', 'or', 'no', 'bleeding', '.', 'we', 'report', 'a', 'case', 'of', 'pulmonary', 'em', '##bol', '##us', 'associated', 'with', 'life', '-', 'threatening', 'di', '##sse', '##minated', 'intra', '##vas', '##cular', 'coa', '##gul', '##ation', '.']
[21908, 7861, 14956, 2271, 1011, 10572, 4487, 11393, 26972, 26721, 12044, 15431, 28155, 24848, 3370, 1012, 21908, 7861, 14956, 2271, 2004, 1037, 3426, 1997, 4487, 11393, 26972, 26721, 12044, 15431, 28155, 24848, 3370, 2038, 2069, 3728, 2042, 3858, 1012, 1996, 19610, 2953, 25032, 22974, 22

Query with the most relevant sensitive documents
	- Document that has the biggest change up
	- Document that has the biggest change down
Query with the least relevant sensitive documents
	- Document that has the biggest change up
	- Document that has the biggest change downe down

In [9]:
# Query with most relevant sensitive documents (53)
max_val = 0

for _, query in pd.merge(qrels, ohsumed_docs).groupby("qid"):
    if query[query["label"] >= 1].sensitivity.sum() > max_val:
        max_val = query[query["label"] >= 1].sensitivity.sum()
        max_qid = query.iloc[0]["qid"]

max_qid

# Query with least relevant sensitive documents (10)
min_val = 10000000000000000000

for _, query in pd.merge(qrels, ohsumed_docs).groupby("qid"):
    if query[query["label"] >= 1].sensitivity.sum() < min_val:
        min_val = query[query["label"] >= 1].sensitivity.sum()
        min_qid = query.iloc[0]["qid"]

min_qid

'10'

In [10]:
relevance = pt.io.read_results("../runs/splade_relevance % 100")
distill = pt.io.read_results("../runs/splade_distil % 100")
combined = pd.merge(relevance, distill, on = ["docno", "qid"])
combined["rank_change"] = combined["rank_x"] - combined["rank_y"] # positive = down from rel to sens. negative = up from rel to sens.
sensitivities = ohsumed_docs[["docno", "sensitivity"]]
combined = pd.merge(combined, sensitivities, on = "docno")
combined = pd.merge(combined, qrels, on = ["docno", "qid"], how = "left")

In [11]:
combined[combined["qid"] == min_qid]

,qid,docno,rank_x,score_x,name_x,rank_y,score_y,name_y,rank_change,sensitivity,label
269,10,310557,0,136457.0,pyterrier,7,105319.0,pyterrier,-7,0,2.0
270,10,111278,1,128949.0,pyterrier,6,108364.0,pyterrier,-5,0,2.0
271,10,347443,4,105354.0,pyterrier,41,84738.0,pyterrier,-37,0,2.0
272,10,6746,16,84920.0,pyterrier,5,109268.0,pyterrier,11,0,2.0
273,10,284324,39,75690.0,pyterrier,4,109730.0,pyterrier,35,0,0.0
274,10,106899,97,62003.0,pyterrier,0,128451.0,pyterrier,97,0,NaN


In [12]:
# Biggest increase rel to sens (153756) rank 95 -> 32 (qid 53)
# Biggest decrease from rel to sens (56730) rank 13 -> 95 (qid 53)
print(f"Query with the most relevant sensitive documnents. ('{queries[queries['qid'] == max_qid]['query'].iloc[0]}')\n")
print("Document with the largest increase in rank when ranked using relevance vs sensitivity SPLADE (rank 95 -> 32).")
print("Relevance SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "153756"].text.iloc[0], relevance_model, "red")
print("Sensitivity SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "153756"].text.iloc[0], distil_model, "red")

print()
print("Document with the largest decrease in rank when ranked using relevance vs sensitivity SPLADE (rank 13 -> 95):")
print("Relevance SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "56730"].text.iloc[0], relevance_model, "red")
print("Sensitivity SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "56730"].text.iloc[0], distil_model, "red")


# Biggest increase from rel to sens (106899) rank 97 -> 0
# Biggest decrease from rel to sense (347443) rank 4 -> 41
print(f"Query with the lease relevant sensitive documnents. ('{queries[queries['qid'] == min_qid]['query'].iloc[0]}')\n")
print("Document with the largest increase in rank when ranked using relevance vs sensitivity SPLADE (rank 97 -> 0):")
print("Relevance SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "106899"].text.iloc[0], relevance_model, "green")
print("Sensitivity SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "106899"].text.iloc[0], distil_model, "green")

print()
print("Document with the largest decrease in rank when ranked using relevance vs sensitivity SPLADE (rank 4 -> 41):")
print("Relevance SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "347443"].text.iloc[0], relevance_model, "green")
print("Sensitivity SPLADE:")

display_colours(ohsumed_docs[ohsumed_docs["docno"] == "347443"].text.iloc[0], distil_model, "green")

Query with the most relevant sensitive documnents. ('lupus nephritis, diagnosis and management')

Document with the largest increase in rank when ranked using relevance vs sensitivity SPLADE (rank 95 -> 32).
Relevance SPLADE:


Sensitivity SPLADE:



Document with the largest decrease in rank when ranked using relevance vs sensitivity SPLADE (rank 13 -> 95):
Relevance SPLADE:


Sensitivity SPLADE:


Query with the lease relevant sensitive documnents. ('effectiveness of gallium therapy for hypercalcemia')

Document with the largest increase in rank when ranked using relevance vs sensitivity SPLADE (rank 97 -> 0):
Relevance SPLADE:


Sensitivity SPLADE:



Document with the largest decrease in rank when ranked using relevance vs sensitivity SPLADE (rank 4 -> 41):
Relevance SPLADE:


Sensitivity SPLADE:
